# Challenge 3 — Gemini + BigQuery: Enhancing Public Communication


## 1. Create the dataset

In [1]:
# Outside a managed BQ notebook, authenticate first:
# from google.colab import auth; auth.authenticate_user()

from google.cloud import bigquery

PROJECT_ID  = None      # e.g. "my-gcp-project"; None => environment default
DATASET     = "weather"
LOCATION    = "US"       # dataset + connection location must match
CONNECTION  = "vertex_conn"   # name for the BigQuery -> Vertex AI connection

client = bigquery.Client(project=PROJECT_ID) if PROJECT_ID else bigquery.Client()
PROJECT_ID = client.project
print("Project:", PROJECT_ID)

RAW_TABLE    = f"{PROJECT_ID}.{DATASET}.weather_data"
REPORT_TABLE = f"{PROJECT_ID}.{DATASET}.weather_reports"
GEMINI_MODEL = f"{PROJECT_ID}.{DATASET}.gemini_model"

def run(sql):
    job = client.query(sql)
    job.result()
    return job


Project: qwiklabs-gcp-01-9ed82c2a1f67


In [2]:
run(f"""
CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.{DATASET}`
OPTIONS(location="{LOCATION}")
""")
print("Dataset ready:", DATASET)


Dataset ready: weather


## 2. Import the CSV into BigQuery


In [3]:
run(f"""
LOAD DATA OVERWRITE `{RAW_TABLE}`
FROM FILES (
  format = 'CSV',
  uris   = ['gs://labs.roitraining.com/data-to-ai-workshop/weather_data.csv'],
  skip_leading_rows = 1,
  field_delimiter = ',',
  max_bad_records = 10
)
""")
print("Loaded:", RAW_TABLE)


Loaded: qwiklabs-gcp-01-9ed82c2a1f67.weather.weather_data


In [4]:
client.query(f"SELECT * FROM `{RAW_TABLE}` LIMIT 10").to_dataframe()

,date,city,state,temperature_f,wind_speed_mph,precipitation_in,barometric_pressure_inHg,humidity_percent,weather_condition
0,2025-02-21,Atlanta,GA,55.7,5.0,0.12,29.80,50.4,Cloudy
1,2025-02-26,Atlanta,GA,75.2,10.4,0.03,29.58,49.9,Cloudy
2,2025-03-01,Atlanta,GA,51.7,4.7,0.08,29.74,49.9,Cloudy
3,2025-03-05,Atlanta,GA,74.4,5.1,0.02,29.92,50.4,Cloudy
4,2025-03-10,Atlanta,GA,59.5,9.6,0.09,29.67,57.2,Cloudy
5,2025-03-14,Atlanta,GA,71.7,7.2,0.18,29.92,55.3,Cloudy
6,2025-02-19,Boston,MA,61.7,3.9,0.11,29.62,54.1,Cloudy
7,2025-03-09,Boston,MA,76.7,4.3,0.09,29.52,40.9,Cloudy
8,2025-03-13,Boston,MA,71.9,9.8,0.16,29.99,42.3,Cloudy
9,2025-03-19,Boston,MA,60.7,6.4,0.04,29.83,49.4,Cloudy


In [5]:
# Size, date span, city/state coverage
client.query(f"""
SELECT
  COUNT(*)                  AS row_count,
  COUNT(DISTINCT city)      AS cities,
  COUNT(DISTINCT state)     AS states,
  MIN(date)                 AS first_date,
  MAX(date)                 AS last_date
FROM `{RAW_TABLE}`
""").to_dataframe()


,row_count,cities,states,first_date,last_date
0,300,10,9,2025-02-18,2025-03-19


In [6]:
# Numeric ranges — sanity-check the readings
client.query(f"""
SELECT
  ROUND(MIN(temperature_f),1) AS min_temp, ROUND(MAX(temperature_f),1) AS max_temp,
  ROUND(MIN(wind_speed_mph),1) AS min_wind, ROUND(MAX(wind_speed_mph),1) AS max_wind,
  ROUND(MIN(precipitation_in),2) AS min_precip, ROUND(MAX(precipitation_in),2) AS max_precip,
  ROUND(MIN(humidity_percent),1) AS min_humid, ROUND(MAX(humidity_percent),1) AS max_humid
FROM `{RAW_TABLE}`
""").to_dataframe()


,min_temp,max_temp,min_wind,max_wind,min_precip,max_precip,min_humid,max_humid
0,14.3,94.4,0.0,49.8,0.0,3.9,20.1,98.6


In [7]:
# Distribution of weather conditions
client.query(f"""
SELECT weather_condition, COUNT(*) AS n,
       ROUND(AVG(temperature_f),1) AS avg_temp,
       ROUND(AVG(wind_speed_mph),1) AS avg_wind
FROM `{RAW_TABLE}`
GROUP BY weather_condition
ORDER BY n DESC
""").to_dataframe()


,weather_condition,n,avg_temp,avg_wind
0,Stormy,72,43.9,34.8
1,Snowy,62,27.2,11.4
2,Sunny,60,81.4,2.8
3,Cloudy,55,63.2,7.2
4,Rainy,51,55.2,11.7


## 4. Set up the Vertex AI connection (one-time)

`ML.GENERATE_TEXT` needs a BigQuery **CLOUD_RESOURCE** connection whose service
account can call Vertex AI. These steps run **once per project** and may require
an admin. Run them in **Cloud Shell** (or any terminal with `gcloud`/`bq`), not
in this notebook, because granting IAM needs project-admin rights.

```bash
# a) Create the connection (location must match your dataset, here: US)
bq mk --connection --location=US --project_id=PROJECT_ID \
   --connection_type=CLOUD_RESOURCE vertex_conn

# b) Find the service account the connection uses
bq show --connection PROJECT_ID.US.vertex_conn
#   -> copy the "serviceAccountId", looks like:
#      bqcx-XXXXXXXX@gcp-sa-bigquery-condel.iam.gserviceaccount.com

# c) Grant that service account permission to call Vertex AI
gcloud projects add-iam-policy-binding PROJECT_ID \
   --member="serviceAccount:bqcx-XXXX@gcp-sa-bigquery-condel.iam.gserviceaccount.com" \
   --role="roles/aiplatform.user"
```

In [8]:
# Helper: print the exact commands with your project filled in to run in cloud shell
print(f"""
bq mk --connection --location={LOCATION} --project_id={PROJECT_ID} \\
   --connection_type=CLOUD_RESOURCE {CONNECTION}

bq show --connection {PROJECT_ID}.{LOCATION}.{CONNECTION}

gcloud projects add-iam-policy-binding {PROJECT_ID} \\
   --member="serviceAccount:<PASTE_SERVICE_ACCOUNT_FROM_ABOVE>" \\
   --role="roles/aiplatform.user"
""")



bq mk --connection --location=US --project_id=qwiklabs-gcp-01-9ed82c2a1f67 \
   --connection_type=CLOUD_RESOURCE vertex_conn

bq show --connection qwiklabs-gcp-01-9ed82c2a1f67.US.vertex_conn

gcloud projects add-iam-policy-binding qwiklabs-gcp-01-9ed82c2a1f67 \
   --member="serviceAccount:<PASTE_SERVICE_ACCOUNT_FROM_ABOVE>" \
   --role="roles/aiplatform.user"



## 5. Create the remote Gemini model


In [10]:
run(f"""
CREATE OR REPLACE MODEL `{GEMINI_MODEL}`
REMOTE WITH CONNECTION `{PROJECT_ID}.{LOCATION}.{CONNECTION}`
OPTIONS (ENDPOINT = 'gemini-2.5-flash')
""")
print("Gemini model registered:", GEMINI_MODEL)


Gemini model registered: qwiklabs-gcp-01-9ed82c2a1f67.weather.gemini_model


## 6. Generate weather reports with `ML.GENERATE_TEXT`


In [13]:
run(f"""
CREATE OR REPLACE TABLE `{REPORT_TABLE}` AS
SELECT
  date, city, state, temperature_f, wind_speed_mph, precipitation_in,
  barometric_pressure_inHg, humidity_percent, weather_condition,
  ml_generate_text_llm_result AS weather_report
FROM ML.GENERATE_TEXT(
  MODEL `{GEMINI_MODEL}`,
  (
    SELECT
      date, city, state, temperature_f, wind_speed_mph, precipitation_in,
      barometric_pressure_inHg, humidity_percent, weather_condition,
      CONCAT(
        'You are a public weather service communicator. Write a concise (2-3 sentence) ',
        'weather report for residents. If conditions are dangerous (high winds, heavy ',
        'precipitation, extreme temperatures, or storms), begin with the single word ',
        'WARNING and clearly state the hazard and a safety tip. Otherwise give a calm, ',
        'friendly summary. Use only the data provided. ',
        'City: ', city, ', ', state, '. ',
        'Date: ', CAST(date AS STRING), '. ',
        'Condition: ', weather_condition, '. ',
        'Temperature: ', CAST(temperature_f AS STRING), ' F. ',
        'Wind: ', CAST(wind_speed_mph AS STRING), ' mph. ',
        'Precipitation: ', CAST(precipitation_in AS STRING), ' in. ',
        'Humidity: ', CAST(humidity_percent AS STRING), ' %. ',
        'Pressure: ', CAST(barometric_pressure_inHg AS STRING), ' inHg.'
      ) AS prompt
    FROM `{RAW_TABLE}`
  ),
  STRUCT(
    0.3 AS temperature,
    512 AS max_output_tokens,
    0.9 AS top_p,
    40  AS top_k,
    TRUE AS flatten_json_output
  )
)
""")
print("Reports written to:", REPORT_TABLE)

Reports written to: qwiklabs-gcp-01-9ed82c2a1f67.weather.weather_reports


## 7. Review the generated reports

In [14]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

client.query(f"""
SELECT city, state, weather_condition, temperature_f, wind_speed_mph, weather_report
FROM `{REPORT_TABLE}`
ORDER BY city
LIMIT 15
""").to_dataframe()


,city,state,weather_condition,temperature_f,wind_speed_mph,weather_report
0,Atlanta,GA,Cloudy,51.7,4.7,"Good morning, Atlanta! Expect a cloudy start to your Saturday with a light breeze. Temperatures will be mild, hovering around 51 degrees Fahrenheit."
1,Atlanta,GA,Cloudy,55.7,5.0,"Good morning, Atlanta! Expect a cloudy day with a light breeze and a comfortable temperature of 55.7°F. We've seen a trace of precipitation overnight, but no significant rain is expected today."
2,Atlanta,GA,Rainy,65.8,6.2,"Good morning, Atlanta! Expect a rainy day with comfortable temperatures around 66 degrees. Winds will be light at 6 mph, so grab your umbrella and stay dry."
3,Atlanta,GA,Rainy,48.3,12.8,"Good morning, Atlanta! Expect a rainy day with temperatures around 48 degrees Fahrenheit and a gentle breeze. Keep an umbrella handy as we're seeing about an inch and a half of rain."
4,Atlanta,GA,Cloudy,75.2,10.4,"Good morning, Atlanta! Expect a cloudy day with comfortable temperatures around 75 degrees and a light breeze. We'll see just a trace of precipitation, so no need for an umbrella."
5,Atlanta,GA,Rainy,55.2,7.9,"Good morning, Atlanta! Expect a rainy day with temperatures around 55 degrees and light winds. Keep an umbrella handy as you head out."
6,Atlanta,GA,Cloudy,59.5,9.6,"Good morning, Atlanta! Expect a cloudy day today with a comfortable temperature of around 60 degrees. We'll see light winds and a very slight chance of sprinkles this morning."
7,Atlanta,GA,Snowy,15.1,17.0,"WARNING: Heavy snow and dangerously cold temperatures are expected today, with winds up to 17 mph making it feel even colder. Limit outdoor exposure and dress in multiple layers if you must go outside."
8,Atlanta,GA,Cloudy,74.4,5.1,"Good morning, Atlanta! Expect a cloudy day with comfortable temperatures around 74 degrees. A light breeze and minimal precipitation will make for a pleasant Wednesday."
9,Atlanta,GA,Snowy,33.4,9.5,"WARNING: Snowy conditions are expected in Atlanta today, with temperatures just above freezing. Roads may be slick, so please drive carefully and allow extra time for your commute."


In [15]:
# How many came back as warnings?
client.query(f"""
SELECT
  COUNTIF(STARTS_WITH(UPPER(TRIM(weather_report)), 'WARNING')) AS warnings,
  COUNT(*) AS total
FROM `{REPORT_TABLE}`
""").to_dataframe()


,warnings,total
0,165,300
